# 06 - Advanced forecasting models

Train a pooled Poisson histogram gradient-boosting model and compare it with Croston's SBA intermittent-demand method. Each validation origin fits only on data available at that date and forecasts recursively for the complete 19-month horizon. Intermittency groups are recalculated from each origin's history; missing targets remain unknown.

The baseline results from notebook 05 are shown alongside these methods. Select a model based on temporal validation, horizon behavior, and relevant demand segments—not training fit.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "features").is_dir():
            return candidate
    raise FileNotFoundError("Run notebooks 01-05 before model backtesting.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.global_model import run_global_model_backtest

summary = run_global_model_backtest(PROJECT_ROOT)
for origin in summary["origins"]:
    print(origin)
print(f"Prediction rows: {summary['prediction_rows']:,}")
print("Leakage controls:")
for control in summary["leakage_controls"]:
    print(f"- {control}")

In [ ]:
import pandas as pd
from IPython.display import display

output_dir = PROJECT_ROOT / "outputs" / "global_model"
overall = pd.read_csv(output_dir / "accuracy_overall.csv")
by_horizon = pd.read_csv(output_dir / "accuracy_by_horizon.csv")
by_intermittency = pd.read_csv(output_dir / "accuracy_by_intermittency.csv")
baseline = pd.read_csv(PROJECT_ROOT / "outputs" / "backtesting" / "accuracy_overall.csv")
baseline = baseline.rename(columns={"Model": "Model"})
comparison = pd.concat(
    [baseline, overall], ignore_index=True, sort=False
).sort_values(["Origin", "WAPE"])

display(comparison[["Origin", "Model", "Rows", "WAPE", "Bias", "Actual_Units", "Forecast_Units"]])
display(by_horizon)
display(by_intermittency)

assert set(overall["Model"]) == {"global_poisson_hgbr", "croston_sba"}
assert set(overall["Origin"]) == {"2023-09", "2025-03"}
assert (overall["Rows"] > 0).all()
assert set(by_intermittency["Intermittency_Class"]).issubset(
    {"smooth", "erratic", "intermittent", "lumpy", "insufficient_positive_history"}
)
print("Validation passed: both models use two complete origins and are compared with notebook 05 baselines.")

Review the intermittent classes separately: Croston SBA is a constant-rate comparator, while the global model can respond to calendar, lagged macro, event, lifecycle, family, and region features. Confirm that gains are consistent across both origins before choosing the production model.